In [1]:

# Load the python libraries for cleaning the Nassau Candy Distributor Dataset
import os
import numpy as np
import pandas as pd
print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
print(os.getcwd())

C:\Users\Admin\Desktop\Unified_Mentor_Hospital_Data


In [6]:
df = pd.read_csv(r"C:\Users\Admin\Desktop\Unified_Mentor_Hospital_Data/HHS_Unaccompanied_Alien_Children_Program.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1170 entries, 0 to 1169
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Date                                             720 non-null    str    
 1   Children apprehended and placed in CBP custody*  720 non-null    float64
 2   Children in CBP custody                          720 non-null    float64
 3   Children transferred out of CBP custody          720 non-null    float64
 4   Children in HHS Care                             720 non-null    str    
 5   Children discharged from HHS Care                720 non-null    float64
dtypes: float64(4), str(2)
memory usage: 69.5 KB


In [7]:
print(20 * "=", "Time-Series-Preparation", "=" * 20)

print("--- Raw Data Status ---")
print(f"Initial shape: {df.shape}")

# 2. Drop completely empty trailing rows
df = df.dropna(how='all').reset_index(drop=True)

# 3. Convert Date column to actual datetime format and sort chronologically
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.sort_values('Date').reset_index(drop=True)

# 4. Clean numeric columns (remove commas from strings like '6,566' and convert to float)
for col in df.columns:
    if col != 'Date':
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# Set Date as the DataFrame index
df.set_index('Date', inplace=True)

# 5. Ensure continuity of daily observations and handle missing days via linear interpolation
df = df.asfreq('D')
df = df.interpolate(method='linear')

# 6. Validation Check
print("\n--- Cleaned & Validated Data Status ---")
print(f"Final shape (Continuous daily observations): {df.shape}")
print("Missing values check per column:")
print(df.isnull().sum())

print("\nFirst 20 rows of cleaned data:")
print(df.head(20))

==================== Time-Series-Preparation ====================
--- Raw Data Status ---
Initial shape: (1170, 6)

--- Cleaned & Validated Data Status ---
Final shape (Continuous daily observations): (1075, 5)
Missing values check per column:
Children apprehended and placed in CBP custody*    0
Children in CBP custody                            0
Children transferred out of CBP custody            0
Children in HHS Care                               0
Children discharged from HHS Care                  0
dtype: int64

First 20 rows of cleaned data:
            Children apprehended and placed in CBP custody*  \
Date                                                          
2023-01-12                                            33.00   
2023-01-13                                            32.90   
2023-01-14                                            32.80   
2023-01-15                                            32.70   
2023-01-16                                            32.60   
2023-

In [8]:
print(20 * "=", "Feature Engineering for Forecasting", "=" * 20)

# Target variable we want to predict/analyze
target = 'Children in HHS Care'

# 2. Lag Features (t-1, t-7, t-14 values)
df['lag_1'] = df[target].shift(1)
df['lag_7'] = df[target].shift(7)
df['lag_14'] = df[target].shift(14)

# 3. Rolling Averages & Variance (7-day and 14-day rolling mean and variance)
df['rolling_mean_7'] = df[target].rolling(window=7).mean()
df['rolling_var_7'] = df[target].rolling(window=7).var()

df['rolling_mean_14'] = df[target].rolling(window=14).mean()
df['rolling_var_14'] = df[target].rolling(window=14).var()

# 4. Flow-Based Signals: Transfers - Discharges (Net Pressure Indicator)
df['net_pressure'] = df['Children transferred out of CBP custody'] - df['Children discharged from HHS Care']

# 5. Calendar Effects: Day of week and month
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
# Weekend Proxy: Flag Saturdays (5) and Sundays (6) where processing volume often drops
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Month-End Proxy: Flag the last 3 days of any month (often administrative crunch times)
df['is_month_end'] = df.index.is_month_end.astype(int)

# Drop rows with NaN values created by shifts/rolling windows (first 13 rows)
df_features = df.dropna().copy()

print("Feature engineering completed successfully!")
print(f"Final shape with features: {df_features.shape}")
print("\nPreview of engineered features:")
print(df_features[['Children in HHS Care', 'net_pressure', 'rolling_mean_7','rolling_var_7','rolling_mean_14','rolling_var_14','day_of_week', 'month', 'is_weekend', 'is_month_end']].tail(5))

==================== Feature Engineering for Forecasting ====================
Feature engineering completed successfully!
Final shape with features: (1061, 17)

Preview of engineered features:
            Children in HHS Care  net_pressure  rolling_mean_7  rolling_var_7  \
Date                                                                            
2025-12-17                2481.0      1.000000     2459.571429     231.989418   
2025-12-18                2472.0    -10.000000     2464.571429     143.656084   
2025-12-19                2476.0     -7.666667     2468.952381      81.349206   
2025-12-20                2480.0     -5.333333     2472.714286      46.238095   
2025-12-21                2484.0     -3.000000     2475.857143      36.809523   

            rolling_mean_14  rolling_var_14  day_of_week  month  is_weekend  \
Date                                                                          
2025-12-17      2445.214286      373.753969            2     12           0   
20

In [9]:
print(20 * "=", "Train–Test Strategy", "=" * 20)

target = 'Children in HHS Care'

# Add features
df['lag_1'] = df[target].shift(1)
df['lag_7'] = df[target].shift(7)
df['lag_14'] = df[target].shift(14)
df['rolling_mean_7'] = df[target].rolling(window=7).mean()
df['rolling_var_7'] = df[target].rolling(window=7).var()
df['net_pressure'] = df['Children transferred out of CBP custody'] - df['Children discharged from HHS Care']
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_month_end'] = df.index.is_month_end.astype(int)

# Drop missing rows from shifts
df_model = df.dropna().copy()

# 2. Strict Time-Based Split (No Random Sampling)
# Training set: 2023 to 2024
train_data = df_model[df_model.index < '2025-01-01']

# Testing set: 2025 onwards
test_data = df_model[df_model.index >= '2025-01-01']

print("Train-Test Split Completed Successfully!")
print(f"Training Samples: {len(train_data)} ({train_data.index.min().date()} to {train_data.index.max().date()})")
print(f"Testing Samples:  {len(test_data)} ({test_data.index.min().date()} to {test_data.index.max().date()})")

# Separate features (X) and target (y)
X_train = train_data.drop(columns=[target])
y_train = train_data[target]

X_test = test_data.drop(columns=[target])
y_test = test_data[target]

==================== Train–Test Strategy ====================
Train-Test Split Completed Successfully!
Training Samples: 706 (2023-01-26 to 2024-12-31)
Testing Samples:  355 (2025-01-01 to 2025-12-21)


In [10]:
print(20 * "=", "Forecasting Models", "=" * 20)

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

target = 'Children in HHS Care'

# Train-Test Split (Train on 2023-2024, Test on 2025)
train = df[df.index < '2025-01-01']
test = df[df.index >= '2025-01-01']

print(20 * "=", "1. BASELINE MODELS", "=" * 20)

# A. Naïve Persistence Model (assumes tomorrow's value equals today's value)
naive_pred = test[target].shift(1)
naive_pred.iloc[0] = train[target].iloc[-1] # fill first test value with last train value

# B. Moving Average Forecast (7-day rolling mean)
# Calculated over the entire series and sliced for the test period
ma_pred = df[target].rolling(window=7).mean().loc[test.index]

# Evaluate Baselines
naive_mae = mean_absolute_error(test[target], naive_pred)
ma_mae = mean_absolute_error(test[target], ma_pred)

print("--- Baseline Model Results ---")
print(f"Naive Persistence Model MAE: {naive_mae:.2f}")
print(f"Moving Average Forecast MAE: {ma_mae:.2f}")


# 2. MACHINE LEARNING MODELS SETUP (Example: Random Forest)

from sklearn.ensemble import RandomForestRegressor

# Re-create features for ML models
df_ml = df.copy()
df_ml['lag_1'] = df_ml[target].shift(1)
df_ml['lag_7'] = df_ml[target].shift(7)
df_ml['rolling_mean_7'] = df_ml[target].rolling(window=7).mean()
df_ml['net_pressure'] = df_ml['Children transferred out of CBP custody'] - df_ml['Children discharged from HHS Care']
df_ml['day_of_week'] = df_ml.index.dayofweek
df_ml['month'] = df_ml.index.month

df_ml = df_ml.dropna()

X_train = df_ml[df_ml.index < '2025-01-01'].drop(columns=[target])
y_train = df_ml[df_ml.index < '2025-01-01'][target]

X_test = df_ml[df_ml.index >= '2025-01-01'].drop(columns=[target])
y_test = df_ml[df_ml.index >= '2025-01-01'][target]

# Train Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_preds)
print("\n--- Machine Learning Model Results ---")
print(f"Random Forest Regressor MAE: {rf_mae:.2f}")

==================== Forecasting Models ====================
==================== 1. BASELINE MODELS ====================
--- Baseline Model Results ---
Naive Persistence Model MAE: 17.99
Moving Average Forecast MAE: 49.78

--- Machine Learning Model Results ---
Random Forest Regressor MAE: 3263.77


In [11]:
#1. Generate predictions (Example using Naïve Persistence Baseline)
naive_preds = test[target].shift(1)
naive_preds.iloc[0] = train[target].iloc[-1] # fill first test value with last training value

# 2. Comprehensive Evaluation Function
def evaluate_forecast(actual, predicted):
    # MAE: Absolute forecast accuracy
    mae = mean_absolute_error(actual, predicted)

    # RMSE: Penalizes large errors (root mean squared error)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    # MAPE: Relative error understanding (percentage)
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100

    return mae, rmse, mape

# Run evaluation on test data
mae_val, rmse_val, mape_val = evaluate_forecast(test[target], naive_preds)

print("--- Model Evaluation Metrics ---")
print(f"MAE  (Mean Absolute Error):          {mae_val:.2f}")
print(f"RMSE (Root Mean Squared Error):      {rmse_val:.2f}")
print(f"MAPE (Mean Absolute Percentage Error): {mape_val:.2f}%")

--- Model Evaluation Metrics ---
MAE  (Mean Absolute Error):          17.99
RMSE (Root Mean Squared Error):      35.79
MAPE (Mean Absolute Percentage Error): 0.56%


In [13]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Assume test_data, y_test, and your model predictions (e.g., naive_preds or ml_preds) are ready
actuals = y_test.values
predictions = naive_preds.values


# Forecast Accuracy (%)
# Calculated based on Mean Absolute Percentage Error (MAPE)
mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100
forecast_accuracy = max(0.0, 100.0 - mape)


# Surge Lead Time
# Logic: Count how many days in advance sudden upward shifts (surges) were successfully flagged
# Let's define a surge threshold (e.g., a daily increase of > 20 children in care load)
actual_diff = np.diff(actuals, prepend=actuals[0])
pred_diff = np.diff(predictions, prepend=predictions[0])

surge_threshold = 20.0
actual_surges = np.where(actual_diff > surge_threshold)[0]
predicted_surges = np.where(pred_diff > surge_threshold)[0]

# Average lead time proxy (mocking or computing overlap window)
if len(actual_surges) > 0 and len(predicted_surges) > 0:
    surge_lead_time_days = max(1, int(np.mean(np.abs(actual_surges[:len(predicted_surges)] - predicted_surges[:len(actual_surges)]))))
else:
    surge_lead_time_days = 7  # Default operational lead time benchmark


# Capacity Breach Probability %
# Logic: Probability (%) that predicted care load exceeds max authorized shelter capacity
max_authorized_capacity = 8500.0  # Example critical capacity threshold
breach_events = np.sum(predictions > max_authorized_capacity)
capacity_breach_probability = (breach_events / len(predictions)) * 100.0


# Forecast Stability Index %
# Logic: Measures model robustness and consistency (low daily error variance = higher stability)
error_series = np.abs(actuals - predictions)
stability_index = max(0.0, 100.0 - (np.std(error_series) / np.mean(actuals) * 100.0))


# Model Robustness Score %
# Logic: Combines overall accuracy and stability across multi-horizon evaluations
rmse = np.sqrt(mean_squared_error(actuals, predictions))
normalized_rmse = rmse / np.mean(actuals)
model_robustness = max(0.0, 100.0 - (normalized_rmse * 100.0))


print(20 * "=", "KPI Report", "=" * 20)
print("=" * 40)
print("       UAC PROGRAM EXECUTIVE KPI REPORT")
print("=" * 40)
print(f"1. Forecast Accuracy:          {forecast_accuracy:.2f}%")
print(f"2. Surge Lead Time:            {surge_lead_time_days} Days")
print(f"3. Capacity Breach Probability: {capacity_breach_probability:.2f}%")
print(f"4. Forecast Stability Index:   {stability_index:.2f}%")
print(f"5. Model Robustness Score:     {model_robustness:.2f}%")
print("=" * 40)

==================== KPI Report ====================
       UAC PROGRAM EXECUTIVE KPI REPORT
1. Forecast Accuracy:          99.44%
2. Surge Lead Time:            1 Days
3. Capacity Breach Probability: 0.00%
4. Forecast Stability Index:   98.78%
5. Model Robustness Score:     98.59%
